In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from PIL import Image, ImageOps
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

In [2]:
DATA_DIR = Path("cnn_image_quiz_3000")
IMG_SIZE = (64, 64)
class_names = ["cat", "dog", "horse"]
VALID_EXT = {".jpg", ".jpeg", ".png"}
splits = ["train", "validation", "test"]
print("Dataset path:", DATA_DIR.resolve())
print("Dataset exists:", DATA_DIR.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Dataset folder not found. Update DATA_DIR "
        "to the correct location of cnn_image_quiz_3000."
    )
for split in splits:
    for cls in class_names:
        folder = DATA_DIR / split / cls

        if not folder.is_dir():
            raise FileNotFoundError(
                f"Required folder is missing: {folder}"
            )

Dataset path: C:\Users\pc\OneDrive\Desktop\DL(image)\cnn_image_quiz_3000
Dataset exists: True


In [3]:
rows = []
bad_images = []
for split in splits:
    for cls in class_names:
        folder = DATA_DIR / split / cls
        all_files = [p for p in folder.iterdir() if p.is_file()]
        valid_files = [
            p for p in all_files
            if p.suffix.lower() in VALID_EXT
        ]
        readable_count = 0
        for file in valid_files:
            try:
                with Image.open(file) as img:
                    img.verify()
                readable_count += 1
            except Exception:
                bad_images.append(str(file))
        rows.append({
            "Split": split,
            "Class": cls,
            "Total files": len(all_files),
            "Supported files": len(valid_files),
            "Readable images": readable_count
        })
count_table = pd.DataFrame(rows)

print(count_table.to_string(index=False))
print("\nUnreadable image files:", len(bad_images))
for file in bad_images:
    print(file)

     Split Class  Total files  Supported files  Readable images
     train   cat          700              700              700
     train   dog          704              704              704
     train horse          700              700              700
validation   cat          150              150              150
validation   dog          150              150              150
validation horse           67               67               67
      test   cat          151              151              151
      test   dog          150              150              150
      test horse          150              150              150

Unreadable image files: 0


In [4]:
def load_split(split):
    X = []
    y = []
    skipped = []
    for label, cls in enumerate(class_names):
        folder = DATA_DIR / split / cls
        for file in sorted(folder.iterdir()):
            if not file.is_file():
                continue
            if file.suffix.lower() not in VALID_EXT:
                continue
            try:
                with Image.open(file) as original:
                    original.verify()
                with Image.open(file) as original:
                    img = ImageOps.exif_transpose(original)
                    img = img.convert("RGB")
                    img = img.resize(
                        IMG_SIZE,
                        Image.Resampling.LANCZOS
                    )
                    arr = np.asarray(img, dtype=np.float32)
                X.append(arr)
                y.append(label)
            except Exception:
                skipped.append(str(file))
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    print(
        f"{split}: loaded {len(X)} images; "
        f"skipped {len(skipped)} files"
    )
    return X, y, skipped
X_train, y_train, bad_train = load_split("train")
X_validation, y_validation, bad_validation = load_split(
    "validation"
)
X_test, y_test, bad_test = load_split("test")
print("\nLoading completed.")

train: loaded 2104 images; skipped 0 files
validation: loaded 367 images; skipped 0 files
test: loaded 451 images; skipped 0 files

Loading completed.


In [5]:
X_train = X_train / 255.0
X_validation = X_validation / 255.0
X_test = X_test / 255.0

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

print("\nTraining labels:", y_train.shape)
print("Validation labels:", y_validation.shape)
print("Test labels:", y_test.shape)

print("\nPixel range:")
print("Minimum:", X_train.min())
print("Maximum:", X_train.max())

print("\nTraining class counts:")
print(np.bincount(y_train, minlength=3))

print("\nValidation class counts:")
print(np.bincount(y_validation, minlength=3))

print("\nTest class counts:")
print(np.bincount(y_test, minlength=3))

Training shape: (2104, 64, 64, 3)
Validation shape: (367, 64, 64, 3)
Test shape: (451, 64, 64, 3)

Training labels: (2104,)
Validation labels: (367,)
Test labels: (451,)

Pixel range:
Minimum: 0.0
Maximum: 1.0

Training class counts:
[700 704 700]

Validation class counts:
[150 150  67]

Test class counts:
[151 150 150]


In [6]:
keras.utils.set_random_seed(RANDOM_STATE)

cnn_model = keras.Sequential([
    keras.layers.Input(shape=(96, 96, 3)),

    # Data augmentation (only active during training)
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),

    # Block 1
    keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    # Block 2
    keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    # Block 3
    keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    # Block 4
    keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    # Classifier
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(3, activation="softmax"),
])

In [7]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Model compiled")

Model compiled


In [8]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1)

Path("cnn_artifacts").mkdir(exist_ok=True)
checkpoint = keras.callbacks.ModelCheckpoint(
    "cnn_artifacts/best_model.keras", monitor="val_loss", save_best_only=True)

history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_validation, y_validation),
    epochs=25, batch_size=32,
    callbacks=[early_stopping, reduce_lr, checkpoint], verbose=1)

Epoch 1/25


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense' is incompatible with the layer: expected axis -1 of input shape to have value 4608, but received input with shape (32, 2048)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 64, 64, 3), dtype=float32)
  • training=True
  • mask=None

In [11]:
Path("cnn_artifacts").mkdir(exist_ok=True)
cnn_model.save("cnn_artifacts/best_model.keras")
print("Saved")

Saved


In [12]:
def show_files(cls, split="test", n=149):
    folder = DATA_DIR / split / cls
    print([p.name for p in sorted(folder.iterdir())[:n]])


In [14]:
test_loss, test_accuracy = cnn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2%}")

test_probabilities = cnn_model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(test_probabilities, axis=1)

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)



Test Loss: 1.4478
Test Accuracy: 35.70%

Classification Report:

              precision    recall  f1-score   support

         cat       0.44      0.13      0.20       151
         dog       0.35      0.94      0.51       150
       horse       1.00      0.01      0.01       150

    accuracy                           0.36       451
   macro avg       0.60      0.36      0.24       451
weighted avg       0.60      0.36      0.24       451



In [15]:

def predict_test_index(index):
    image = X_test[index]
    actual_label = class_names[y_test[index]]

    probabilities = cnn_model.predict(
        image[np.newaxis, ...],
        verbose=0
    )[0]

    predicted_label = class_names[np.argmax(probabilities)]
    print("Actual:", actual_label)
    print("Predicted:", predicted_label)

    for cls, probability in zip(class_names, probabilities):
        print(f"{cls}: {probability:.2%}")


predict_test_index(0)

Actual: cat
Predicted: dog
cat: 20.21%
dog: 73.47%
horse: 6.32%


In [16]:
import json

report = classification_report(
    y_test, y_pred, labels=[0, 1, 2],
    target_names=class_names, output_dict=True, zero_division=0)

metrics = {
    "test_accuracy": float(test_accuracy * 100),
    "test_images": int(len(y_test)),
    "train_images": int(len(y_train)),
    "report": {c: report[c] for c in class_names},
}
Path("cnn_artifacts").mkdir(exist_ok=True)
Path("cnn_artifacts/metrics.json").write_text(json.dumps(metrics, indent=2))

550

In [22]:
def prepare_image(image_path, mode="crop"):
    with Image.open(image_path) as original:
        img = ImageOps.exif_transpose(original).convert("RGB")
    if mode == "crop":
        img = ImageOps.fit(img, IMG_SIZE, Image.Resampling.LANCZOS)
    else:   # "crop_blur": looks more like the small training images
        img = ImageOps.fit(img, (32, 32), Image.Resampling.LANCZOS).resize(IMG_SIZE, Image.Resampling.LANCZOS)
    return np.asarray(img, dtype=np.float32) / 255.0


def predict_new_image(image_path):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path.resolve()}")

    batch = np.stack([prepare_image(image_path, "crop"),
                      prepare_image(image_path, "crop_blur")])
    probs = cnn_model.predict(batch, verbose=0).mean(axis=0)   # average of both versions
    best = int(np.argmax(probs))

    if probs[best] < 0.60:
        print(f"Prediction: not sure (best guess {class_names[best]}, {probs[best]:.0%})")
    else:
        print(f"Prediction: {class_names[best]} ({probs[best]:.0%})")

predict_new_image(r"C:\Users\pc\OneDrive\Pictures\Vedios\cat5.png")

Prediction: not sure (best guess cat, 57%)
